# Itération 3 — la classe « Autre »

**Projet Big Data & Deep Learning — Agent d'audit de conformité documentaire**

## Le problème trouvé par l'essai de bout en bout

Sur 8 contrats réels, le système (découpage + Legal-BERT itération 2) retrouve 66 % des clauses repérées par
les juristes, mais **annonce une catégorie sur 100 % des paragraphes neutres** (définitions, signatures,
formules de politesse) : Legal-BERT n'a que 41 réponses possibles, donc il en choisit toujours une. Un seuil
de confiance ne règle pas cela (à 0,5, il reste 55 % de « fantômes »).

## Ce qu'on change

Une **42e réponse : « Autre »**. On ajoute à l'entraînement des paragraphes neutres tirés de contrats
d'entraînement (2 000), et au test des paragraphes neutres de contrats de test (500) : jamais les mêmes contrats.
Tout le reste est identique à l'itération 2 : même modèle, poids de classes, arrêt automatique, graine.

## Mode d'emploi

1. **Exécution** > **Modifier le type d'exécution** > **T4 GPU** > Enregistrer
2. **Exécution** > **Tout exécuter**
3. Cellule 3 : dépose les **QUATRE** fichiers de `C:\projets\audit-conformite-mlops\data\processed` :
   `clauses_train.jsonl`, `clauses_test.jsonl`, `autre_train.jsonl`, `autre_test.jsonl`

Durée : **20 à 30 minutes**.

> Si Colab **redémarre tout seul** après la cellule d'installation, c'est normal : relance à partir de la cellule suivante. Les messages d'erreur rouges de pip à propos de bibliothèques que l'on n'utilise pas (diffusers, gradio...) sont sans importance.

## 1. GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU détecté :", torch.cuda.get_device_name(0))
else:
    print("AUCUN GPU. Exécution > Modifier le type d'exécution > T4 GPU, puis relance.")

## 2. Installation

In [ ]:
import sys
print("Python", sys.version.split()[0])
!pip install "transformers==4.46.3" "tokenizers==0.20.3" "huggingface-hub<1.0" "accelerate>=1.1,<2" scikit-learn 2>&1 | tail -6

In [ ]:
# Vérifie que la bonne version est INSTALLÉE et CHARGÉE (Colab fournit transformers 5, qui est incompatible
# avec ce notebook : une version déjà chargée en mémoire masquerait l'installation).
import sys, os, importlib.metadata as md

on_disk = md.version("transformers")
loaded = sys.modules["transformers"].__version__ if "transformers" in sys.modules else None
print(f"transformers installé : {on_disk}  |  chargé en mémoire : {loaded}")
if on_disk != "4.46.3":
    raise SystemExit("La version 4.46.3 n'a pas pu être installée : copie l'affichage de la cellule précédente.")
if loaded not in (None, "4.46.3"):
    print("\nLa session utilisait une autre version : redémarrage automatique (normal).")
    print("-> Quand Colab a redémarré, relance à partir de la CELLULE 3 (pas besoin de refaire celle-ci).")
    os.kill(os.getpid(), 9)

# Test d'import immédiat : s'il y a un problème, on le voit ici en 5 secondes, pas après 25 minutes.
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback)
print("Imports OK : transformers", __import__("transformers").__version__)

## 3. Dépôt des données

Sélectionne les **QUATRE** fichiers d'un coup.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
NEEDED = ("clauses_train.jsonl", "clauses_test.jsonl", "autre_train.jsonl", "autre_test.jsonl")
for name in NEEDED:
    if os.path.exists(name):
        n = sum(1 for _ in open(name, encoding="utf-8"))
        print(f"OK   {name:22s} {n:>7,} lignes")
    else:
        print(f"MANQUANT : {name}  -> relance cette cellule et choisis les bons fichiers")

## 4. Données : train, validation, test

On ajoute les paragraphes « Autre » aux clauses. Le test contient les 1 912 clauses habituelles **plus** 500 paragraphes neutres de contrats de test.

In [ ]:
import json
import numpy as np
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

NEUTRAL = "Autre"

def load_split(path):
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("label"):
                texts.append(row["text"]); labels.append(row["label"])
    return texts, labels

Xc, yc = load_split("clauses_train.jsonl");  Xa, ya = load_split("autre_train.jsonl")
Xct, yct = load_split("clauses_test.jsonl"); Xat, yat = load_split("autre_test.jsonl")
X_full, y_full_raw = Xc + Xa, yc + ya
X_test, y_test_raw = Xct + Xat, yct + yat

encoder = LabelEncoder().fit(y_full_raw)
known = set(encoder.classes_)
keep = [i for i, l in enumerate(y_test_raw) if l in known]
X_test = [X_test[i] for i in keep]
y_test = encoder.transform([y_test_raw[i] for i in keep]).tolist()
y_full = encoder.transform(y_full_raw).tolist()
n_labels = len(encoder.classes_)
AUTRE_ID = int(encoder.transform([NEUTRAL])[0])
CLAUSE_IDS = [i for i in range(n_labels) if i != AUTRE_ID]

counts = Counter(y_full)
strat = y_full if min(counts.values()) >= 2 else None       # comme à l'itération 2
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.10, random_state=42, stratify=strat)

# Catégories rares : moins de 100 exemples d'entraînement (comme à l'itération 2).
RARE = sorted(c for c, n in counts.items() if n < 100 and c != AUTRE_ID)

print(f"train      : {len(X_train):,}  (dont « Autre » : {sum(1 for y in y_train if y == AUTRE_ID):,})")
print(f"validation : {len(X_val):,}")
print(f"test       : {len(X_test):,}  (dont « Autre » : {sum(1 for y in y_test if y == AUTRE_ID):,})")
print(f"classes    : {n_labels}  (41 catégories + « Autre », dont {len(RARE)} rares)")
assert n_labels == 42, "On attend 42 classes : 41 catégories + Autre."

## 5. Poids des classes

Même formule qu'à l'itération 2 : racine de (effectif moyen / effectif de la classe). « Autre », très fréquente, reçoit un poids faible.

In [ ]:
train_counts = np.bincount(y_train, minlength=n_labels).astype(float)
train_counts[train_counts == 0] = 1.0
weights = np.sqrt(train_counts.mean() / train_counts)
weights = weights / weights.mean()
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float)

print(f"Poids de « Autre » : {weights[AUTRE_ID]:.2f}")
order = np.argsort(weights)
print("Poids les plus forts (catégories rares) :")
for i in order[-3:]:
    print(f"   {encoder.classes_[i][:40]:40s} {weights[i]:.2f}")

## 6. Configuration

In [ ]:
MODELS = [
    {"key": "legalbert-v3", "name": "nlpaueb/legal-bert-base-uncased", "label": "Legal-BERT v3 (avec Autre)",
     "fp16": True, "lr": 3e-5},
]
MAX_EPOCHS = 10
PATIENCE   = 2
BATCH_SIZE = 32
MAX_LENGTH = 256
SEED       = 42

# Référence : itération 2 (41 classes, mesurée sur les 1 912 clauses du test)
V2_F1_MACRO, V2_F1_RARE, BASELINE_F1 = 0.719, 0.638, 0.670
RESULTS_FILE = "benchmark_v3.json"
print(f"{len(MODELS)} modèle | jusqu'à {MAX_EPOCHS} époques | arrêt après {PATIENCE} époques sans progrès")

## 7. Entraînement pondéré avec arrêt automatique

In [ ]:
import gc, time, shutil
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback, set_seed)
from sklearn.metrics import f1_score, accuracy_score


class ClauseDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc, self.labels = enc, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item


class WeightedTrainer(Trainer):
    """Trainer dont la perte pénalise davantage les erreurs sur les classes rares."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(outputs.logits.device))
        loss = loss_fn(outputs.logits.float(), labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"f1_macro": f1_score(p.label_ids, preds, average="macro", zero_division=0)}


def run_model(cfg):
    set_seed(SEED)
    print(f"\n{'=' * 64}\n  {cfg['label']}  ({cfg['name']})\n{'=' * 64}")
    tok = AutoTokenizer.from_pretrained(cfg["name"])
    enc = lambda X: tok(X, truncation=True, padding="max_length", max_length=MAX_LENGTH)
    ds_tr, ds_va, ds_te = (ClauseDataset(enc(X), y) for X, y in ((X_train, y_train), (X_val, y_val), (X_test, y_test)))

    model = AutoModelForSequenceClassification.from_pretrained(cfg["name"], num_labels=n_labels)
    out_dir = f"ckpt_{cfg['key']}"
    args = TrainingArguments(
        output_dir=out_dir, num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=cfg["lr"], warmup_ratio=0.1, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
        logging_steps=50, fp16=cfg["fp16"], report_to=[], seed=SEED,
    )
    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )

    t0 = time.time()
    trainer.train()
    train_s = time.time() - t0

    evals = [h for h in trainer.state.log_history if "eval_f1_macro" in h]
    best_epoch = int(max(evals, key=lambda h: h["eval_f1_macro"])["epoch"])
    epochs_run = int(evals[-1]["epoch"])

    # Le modèle est sauvegardé AVANT les mesures : une erreur de mesure ne fait pas perdre l'entraînement.
    trainer.save_model(f"model_{cfg['key']}")
    tok.save_pretrained(f"model_{cfg['key']}")

    # Évaluation finale sur le TEST, jamais vu pendant l'entraînement.
    t0 = time.time()
    preds = np.argmax(trainer.predict(ds_te).predictions, axis=-1)
    infer_s = time.time() - t0
    y_arr = np.array(y_test)

    # (a) Comparable à l'itération 2 : les lignes qui sont de vraies clauses, 41 catégories.
    is_clause = y_arr != AUTRE_ID
    yt, yp = y_arr[is_clause], preds[is_clause]
    f1_41 = f1_score(yt, yp, labels=sorted(set(yt.tolist())), average="macro", zero_division=0)
    rare_mask = np.isin(yt, RARE)
    f1_rare = f1_score(yt[rare_mask], yp[rare_mask], labels=RARE, average="macro", zero_division=0)
    acc_41 = accuracy_score(yt, yp)

    # (b) Détection du neutre : « Autre » est-elle reconnue, sans manger les vraies clauses ?
    true_a, pred_a = y_arr == AUTRE_ID, preds == AUTRE_ID
    tp = int((true_a & pred_a).sum())
    autre_recall = tp / max(int(true_a.sum()), 1)               # neutres bien reconnus
    autre_precision = tp / max(int(pred_a.sum()), 1)
    clauses_perdues = float(((yp == AUTRE_ID)).mean())          # vraies clauses classées « Autre » à tort

    res = {
        "key": cfg["key"], "label": cfg["label"], "model_name": cfg["name"], "iteration": 3,
        "f1_macro": float(f1_41),                  # 41 catégories, comparable à 0,719
        "f1_macro_rare": float(f1_rare), "accuracy": float(acc_41),
        "f1_macro_42_classes": float(f1_score(y_arr, preds, average="macro", zero_division=0)),
        "autre_rappel": float(autre_recall), "autre_precision": float(autre_precision),
        "fantomes_classification": float(1 - autre_recall),
        "clauses_classees_autre": clauses_perdues,
        "train_minutes": round(train_s / 60, 2),
        "latency_ms_per_clause": round(1000 * infer_s / len(X_test), 2),
        "params_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
        "best_epoch": best_epoch, "epochs_run": epochs_run, "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE, "learning_rate": cfg["lr"], "batch_size": BATCH_SIZE,
        "max_length": MAX_LENGTH, "fp16": cfg["fp16"], "seed": SEED,
        "class_weighting": "sqrt_inverse_frequency",
        "n_train": len(X_train), "n_val": len(X_val), "n_test": len(X_test),
        "n_autre_train": int(sum(1 for y in y_train if y == AUTRE_ID)),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }
    np.save(f"preds_{cfg['key']}.npy", preds)
    shutil.rmtree(out_dir, ignore_errors=True)

    print(f"\n  41 catégories : F1 macro {res['f1_macro']:.3f} | F1 rares {res['f1_macro_rare']:.3f} | "
          f"accuracy {res['accuracy']:.3f}   (itération 2 : {V2_F1_MACRO:.3f})")
    print(f"  « Autre »     : rappel {autre_recall:.3f} | précision {autre_precision:.3f} | "
          f"vraies clauses perdues dans « Autre » : {100 * clauses_perdues:.1f} %")
    print(f"  meilleure époque {best_epoch}/{epochs_run}")
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    return res


results = json.load(open(RESULTS_FILE)) if os.path.exists(RESULTS_FILE) else []
done = {r["key"] for r in results}
for cfg in MODELS:
    if cfg["key"] in done:
        continue
    try:
        results.append(run_model(cfg))
        json.dump(results, open(RESULTS_FILE, "w"), indent=2)
    except Exception as e:
        print(f"\nÉCHEC sur {cfg['label']} : {type(e).__name__}: {e}")
        gc.collect(); torch.cuda.empty_cache()

print(f"\n{len(results)} / {len(MODELS)} modèle terminé.")

## 8. Verdict : itération 2 contre itération 3

In [ ]:
best = max(results, key=lambda r: r["f1_macro"])
d = best["f1_macro"] - V2_F1_MACRO
print("=" * 70)
print(f"  Baseline TF-IDF + SVM        : F1 macro {BASELINE_F1:.3f}")
print(f"  Itération 2 (41 classes)     : F1 macro {V2_F1_MACRO:.3f}   rares {V2_F1_RARE:.3f}")
print(f"  Itération 3 (avec « Autre ») : F1 macro {best['f1_macro']:.3f}   rares {best['f1_macro_rare']:.3f}   ({d:+.3f})")
print("-" * 70)
print(f"  « Autre » : rappel {best['autre_rappel']:.3f}  |  paragraphes neutres encore pris pour une catégorie : "
      f"{100 * best['fantomes_classification']:.1f} %")
print(f"              vraies clauses perdues dans « Autre » : {100 * best['clauses_classees_autre']:.1f} %")
print("=" * 70)
print("Le vrai verdict vient de l'essai de bout en bout sur de vrais contrats :")
print("   python scripts/eval_pipeline.py --model models/best_model_v3 --contracts 8")
print("(l'itération 2 donnait : rappel 66 %, fantômes 55 % au seuil 0,5)")

## 9. Téléchargement

Le modèle est zippé avec ses étiquettes (42, dont « Autre ») et ses métriques.

In [ ]:
shutil.rmtree("best_model_v3", ignore_errors=True)
shutil.copytree(f"model_{best['key']}", "best_model_v3")
json.dump(list(encoder.classes_), open("best_model_v3/labels.json", "w"), indent=2, ensure_ascii=False)
json.dump(best, open("best_model_v3/metrics.json", "w"), indent=2)
!zip -qr best_model_v3.zip best_model_v3

for f in ("benchmark_v3.json", "best_model_v3.zip"):
    files.download(f)